# PH2_NB6e — Eleven New Statistical Features (Vstat: R^5 -> R^16)

The feature ablation showed that only two of the current five features (burstiness and TTR) carry
real weight on the frozen-hybrid LOGO, while targeted perplexity, entity density and discourse
coherence sit near-inert. That leaves room: this notebook computes **eleven additional statistical
features** on the same corpus and merges them with the existing five into a single 16-feature file.

**Inputs:** `aigt-dataset` (dataset.parquet) and `aigt-vstat` (vstat_scaled.parquet, the existing
five, already z-scored).
**Output:** `vstat16_scaled.parquet` — 16 features per article, the eleven new ones z-scored with a
scaler fit on the **train split only** (same leakage discipline as NB4), plus `vstat11_raw.parquet`
for inspection and `scaler11.pkl`.

## The eleven, and why each is not a duplicate of what we already have

| # | Feature | Signal it captures | Distinct from the existing five because |
|---|---------|--------------------|------------------------------------------|
| 1 | `quote_ratio` | share of words inside quotation marks | direct evidence from corpus construction: quote usage ranged 99% (GPT) to 29% (Gemini) vs 92% human |
| 2 | `quote_variety` | how many distinct quote-mark styles appear | typography habit, orthogonal to content |
| 3 | `coord_sub_ratio` | parataxis (wa-, fa-, thumma) vs explicit subordination | Arabic-specific **syntax**; nothing existing measures clause linkage |
| 4 | `function_word_ratio` | closed-class words / all tokens | TTR measures type variety, not the function/content balance |
| 5 | `pos_entropy` | entropy of the part-of-speech distribution | grammatical regularity, not lexical or length-based |
| 6 | `passive_ratio` | frequency of passive / impersonal constructions | voice preference; human reporters favour active attribution |
| 7 | `clause_depth` | mean clauses per sentence (subordination-nesting proxy) | structural complexity **within** a sentence; burstiness only sees sentence length |
| 8 | `compressibility` | gzip size / raw size | predictability **without** a surrogate LM — mathematically independent of targeted PPL |
| 9 | `char_ngram_repetition` | repeated character 4-grams | surface, sub-word repetition; discourse coherence measures semantic flow |
| 10 | `zipf_deviation` | slope of log-rank vs log-frequency, minus the natural -1 | shape of the whole frequency distribution, not a single ratio |
| 11 | `sent_opener_diversity` | unique sentence-opening words / sentences | positional templating ("wa-fi hatha al-siyaq..."), unmeasured so far |

## Honest notes on two proxies

- **`pos_entropy`** uses CAMeL Tools' morphological disambiguator when it loads. If the data package
  is unavailable, the notebook falls back to a coarse word-shape tag set and records which path ran
  in `pos_source`. Report whichever path produced the final numbers.
- **`passive_ratio`** and **`clause_depth`** are *proxies*, not parser output. Undiacritized Arabic
  hides the internal-vowel passive, so `passive_ratio` counts the productive analytic constructions
  (tamma / jara + masdar) and impersonal forms; `clause_depth` counts clause delimiters and
  subordinators rather than parsing a dependency tree. Both are deterministic and reproducible, and
  the thesis should describe them as proxies.

Every feature is computed independently and written incrementally, so one failing extractor never
costs the others.

## Setup

In [1]:
!pip -q install camel-tools >/dev/null 2>&1

import pandas as pd, numpy as np, re, os, glob, gzip, math, time, pickle
from collections import Counter

SEED = 42
np.random.seed(SEED)
OUT_DIR = '/kaggle/working'

def find_file(preferred, pattern, *keywords):
    if os.path.exists(preferred): return preferred
    for kw in keywords:
        hits = [p for p in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    raise FileNotFoundError(preferred)

DATA  = find_file('/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet', '*.parquet', 'dataset')
VSTAT = find_file('/kaggle/input/notebooks/bahaaqassem/nb4-extract-vstat/vstat_scaled.parquet', '*.parquet', 'vstat_scaled', 'vstat')

df = pd.read_parquet(DATA)
vs5 = pd.read_parquet(VSTAT)
OLD5 = ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']

# alignment contract, same as everywhere in this project
vs5_al = vs5.set_index('article_id').loc[df['article_id']].reset_index()
assert (vs5_al['article_id'].to_numpy() == df['article_id'].to_numpy()).all(), 'vstat misalignment'
assert (vs5_al['label'].to_numpy() == df['label'].to_numpy()).all(), 'label mismatch'
print('ALIGNMENT OK | dataset', df.shape, '| existing vstat', vs5_al.shape)
print('splits:', df['split'].value_counts().to_dict())

ALIGNMENT OK | dataset (7101, 7) | existing vstat (7101, 9)
splits: {'train': 5363, 'test': 1093, 'val': 645}


## Shared text utilities

The sentence splitter and whitespace tokenizer are byte-identical to NB4's, so sentence-level
features here are measured on exactly the same segmentation as burstiness.

In [2]:
_SENT_SPLIT = re.compile(r'[.!?\u061f\u0964\n]+')

def split_sentences(text):
    return [s.strip() for s in _SENT_SPLIT.split(text) if s.strip()]

def tokenize_ws(text):
    return [t for t in re.split(r'\s+', text.strip()) if t]

AR_DIAC = re.compile(r'[\u064b-\u0652\u0670\u0640]')
def strip_diac(t):
    return AR_DIAC.sub('', t)

print('text utilities ready')

text utilities ready


## Features 1-2 — Quotation behaviour

Corpus construction already showed quote usage is one of the strongest per-generator fingerprints
(99% of GPT articles contain quotes, 29% of Gemini's, 92% of human). `quote_ratio` measures how much
of the article sits inside quotation marks; `quote_variety` counts how many distinct quote-mark
conventions the writer mixes.

In [3]:
QUOTE_PAIRS = [('\u00ab', '\u00bb'), ('\u201c', '\u201d'), ('"', '"'), ("'", "'")]
QUOTE_CHARS = '\u00ab\u00bb\u201c\u201d\u201e\u201f"\''

def quote_ratio(text):
    toks = tokenize_ws(text)
    if not toks: return float('nan')
    inside = 0
    for op, cl in QUOTE_PAIRS:
        if op == cl:
            parts = text.split(op)
            for k in range(1, len(parts), 2):
                inside += len(tokenize_ws(parts[k]))
        else:
            for m in re.finditer(re.escape(op) + r'(.*?)' + re.escape(cl), text, flags=re.S):
                inside += len(tokenize_ws(m.group(1)))
    return float(min(inside, len(toks))) / len(toks)

def quote_variety(text):
    styles = 0
    for op, cl in QUOTE_PAIRS:
        if op == cl:
            if text.count(op) >= 2: styles += 1
        elif op in text and cl in text:
            styles += 1
    return float(styles)

print('quote features ready')

quote features ready


## Features 3-4 — Clause linkage and function words

`coord_sub_ratio` contrasts Arabic parataxis (chaining with wa-, fa-, thumma) against explicit
subordination. Human news Arabic leans heavily on coordination; LLM prose tends to spell out logical
relations. `function_word_ratio` is the classic stylometric closed-class ratio, which TTR does not
capture (TTR counts type variety, not the function/content balance).

In [4]:
SUBORDINATORS = ['\u0627\u0644\u0630\u064a', '\u0627\u0644\u062a\u064a', '\u0627\u0644\u0630\u064a\u0646',
    '\u0627\u0644\u0644\u0627\u062a\u064a', '\u0644\u0623\u0646', '\u0644\u0643\u0648\u0646',
    '\u062d\u064a\u062b', '\u0628\u064a\u0646\u0645\u0627', '\u0625\u0630\u0627', '\u0625\u0646',
    '\u0623\u0646', '\u0639\u0646\u062f\u0645\u0627', '\u062d\u062a\u0649', '\u0643\u064a',
    '\u0644\u0643\u064a', '\u0645\u0627 \u062f\u0627\u0645', '\u0631\u063a\u0645',
    '\u0628\u0627\u0644\u0631\u063a\u0645', '\u0645\u0646\u0630 \u0623\u0646', '\u0628\u0639\u062f\u0645\u0627',
    '\u0642\u0628\u0644 \u0623\u0646', '\u0645\u0645\u0627', '\u0627\u0644\u0630\u064a\u0646']
COORD_WORDS = ['\u062b\u0645', '\u0623\u0648', '\u0628\u0644', '\u0644\u0643\u0646', '\u0641\u0642\u062f']

def coord_sub_ratio(text):
    toks = [strip_diac(t) for t in tokenize_ws(text)]
    if not toks: return float('nan')
    # coordination: standalone coordinators + word-initial waw/faa on longer words
    coord = sum(1 for t in toks if t in COORD_WORDS)
    coord += sum(1 for t in toks if len(t) > 2 and t[0] in '\u0648\u0641')
    sub = sum(text.count(s) for s in SUBORDINATORS)
    denom = coord + sub
    if denom == 0: return float('nan')
    return float(coord) / float(denom)

FUNCTION_WORDS = set(['\u0641\u064a', '\u0645\u0646', '\u0625\u0644\u0649', '\u0639\u0644\u0649',
    '\u0639\u0646', '\u0645\u0639', '\u0628\u064a\u0646', '\u0639\u0646\u062f', '\u0644\u062f\u0649',
    '\u062e\u0644\u0627\u0644', '\u0628\u0639\u062f', '\u0642\u0628\u0644', '\u0645\u0646\u0630',
    '\u062d\u062a\u0649', '\u0625\u0646', '\u0623\u0646', '\u0623\u0646\u0647', '\u0625\u0646\u0647',
    '\u0627\u0644\u0630\u064a', '\u0627\u0644\u062a\u064a', '\u0647\u0630\u0627', '\u0647\u0630\u0647',
    '\u0630\u0644\u0643', '\u062a\u0644\u0643', '\u0647\u0648', '\u0647\u064a', '\u0647\u0645',
    '\u0647\u0646', '\u0646\u062d\u0646', '\u0623\u0646\u0627', '\u0643\u0644', '\u0628\u0639\u0636',
    '\u063a\u064a\u0631', '\u0644\u0627', '\u0645\u0627', '\u0644\u0645', '\u0644\u0646',
    '\u0642\u062f', '\u0644\u0642\u062f', '\u0623\u064a', '\u0623\u064a\u0636\u0627',
    '\u0643\u0645\u0627', '\u0644\u0643\u0646', '\u0623\u0648', '\u062b\u0645', '\u0628\u0644',
    '\u0625\u0630', '\u0625\u0630\u0627', '\u0644\u0623\u0646', '\u062d\u064a\u062b'])

def function_word_ratio(text):
    toks = [strip_diac(t) for t in tokenize_ws(text)]
    if not toks: return float('nan')
    return float(sum(1 for t in toks if t in FUNCTION_WORDS)) / len(toks)

print('linkage + function-word features ready')

linkage + function-word features ready


## Feature 5 — Part-of-speech entropy

Entropy of the POS distribution: a text that leans on a narrow set of grammatical categories scores
lower. CAMeL Tools' MLE disambiguator provides the tags when its data package is available; the
fallback is a coarse word-shape tagger. `POS_SOURCE` records which ran, and belongs in the
methodology note.

In [5]:
POS_SOURCE = 'fallback'
_camel_disambig = None
try:
    from camel_tools.disambig.mle import MLEDisambiguator
    _camel_disambig = MLEDisambiguator.pretrained()
    POS_SOURCE = 'camel_mle'
    print('POS tagger: CAMeL Tools MLE disambiguator')
except Exception as e:
    print(f'POS tagger: falling back to word-shape tags ({type(e).__name__})')

AL = '\u0627\u0644'
VERB_PREFIX = '\u064a\u062a\u0646\u0623'

def _shape_tags(toks):
    tags = []
    for t in toks:
        t = strip_diac(t)
        if t in FUNCTION_WORDS:        tags.append('FUNC')
        elif t.startswith(AL):          tags.append('DEF_NOUN')
        elif t and t[0] in VERB_PREFIX: tags.append('VERB_LIKE')
        elif t.isdigit():               tags.append('NUM')
        else:                           tags.append('OTHER')
    return tags

def pos_entropy(text):
    toks = tokenize_ws(text)
    if len(toks) < 20: return float('nan')
    if _camel_disambig is not None:
        try:
            sample = toks[:600]                       # cap for speed; distribution is stable
            disambig = _camel_disambig.disambiguate(sample)
            tags = [d.analyses[0].analysis.get('pos', 'na') if d.analyses else 'na'
                    for d in disambig]
        except Exception:
            tags = _shape_tags(toks)
    else:
        tags = _shape_tags(toks)
    counts = Counter(tags)
    total = sum(counts.values())
    if total == 0: return float('nan')
    probs = [c/total for c in counts.values()]
    return float(-sum(p * math.log(p + 1e-12) for p in probs))

print('POS entropy ready | source =', POS_SOURCE)

POS tagger: falling back to word-shape tags (FileNotFoundError)
POS entropy ready | source = fallback


## Features 6-7 — Voice and clause structure (documented proxies)

Undiacritized Arabic hides the internal-vowel passive, so `passive_ratio` counts the productive
analytic passives and impersonal constructions instead. `clause_depth` approximates syntactic
embedding by counting clause delimiters and subordinators per sentence rather than parsing.

In [6]:
PASSIVE_MARKERS = ['\u062a\u0645 ', '\u062a\u0645\u062a ', '\u064a\u062a\u0645 ',
                   '\u062c\u0631\u0649 ', '\u062c\u0631\u062a ', '\u062a\u0639\u0631\u0636',
                   '\u062a\u0645\u062a\u0627 ', '\u064a\u062c\u0631\u064a ']

def passive_ratio(text):
    sents = split_sentences(text)
    if not sents: return float('nan')
    hits = sum(text.count(m) for m in PASSIVE_MARKERS)
    return float(hits) / len(sents)

CLAUSE_DELIMS = ['\u060c', ';', '\u061b', ' - ', ' \u2013 ']

def clause_depth(text):
    sents = split_sentences(text)
    if not sents: return float('nan')
    total = 0
    for s in sents:
        delims = sum(s.count(d) for d in CLAUSE_DELIMS)
        subs = sum(s.count(x) for x in SUBORDINATORS)
        total += 1 + delims + subs
    return float(total) / len(sents)

print('voice + clause features ready')

voice + clause features ready


## Features 8-10 — Compression, sub-word repetition, and frequency shape

`compressibility` is the one predictability measure in this set that needs no surrogate language
model, so it is mathematically independent of targeted perplexity. `char_ngram_repetition` catches
surface formulaic repetition below the word level. `zipf_deviation` describes the shape of the whole
word-frequency distribution rather than any single ratio.

In [7]:
def compressibility(text):
    raw = text.encode('utf-8')
    if len(raw) < 200: return float('nan')
    comp = gzip.compress(raw, compresslevel=6)
    return float(len(comp)) / float(len(raw))

def char_ngram_repetition(text, n=4):
    s = re.sub(r'\s+', ' ', text)
    if len(s) < n * 20: return float('nan')
    grams = [s[i:i+n] for i in range(len(s) - n + 1)]
    if not grams: return float('nan')
    return 1.0 - (len(set(grams)) / len(grams))

def zipf_deviation(text):
    toks = [strip_diac(t) for t in tokenize_ws(text)]
    if len(toks) < 100: return float('nan')
    freqs = sorted(Counter(toks).values(), reverse=True)
    freqs = [f for f in freqs if f > 0]
    if len(freqs) < 20: return float('nan')
    ranks = np.arange(1, len(freqs) + 1)
    slope, _ = np.polyfit(np.log(ranks), np.log(np.array(freqs, dtype=float)), 1)
    return float(slope + 1.0)          # 0 == textbook Zipf

print('compression / repetition / zipf features ready')

compression / repetition / zipf features ready


## Feature 11 — Sentence-opener diversity

LLM long-form text recycles transitional openers. This counts how many distinct words start the
sentences of an article, normalized by sentence count. It is positional, so it is not what discourse
coherence measures (marker recurrence anywhere in the text).

In [8]:
def sent_opener_diversity(text):
    sents = split_sentences(text)
    if len(sents) < 3: return float('nan')
    openers = []
    for s in sents:
        toks = tokenize_ws(s)
        if toks: openers.append(strip_diac(toks[0]))
    if not openers: return float('nan')
    return float(len(set(openers))) / float(len(openers))

NEW11 = ['quote_ratio', 'quote_variety', 'coord_sub_ratio', 'function_word_ratio',
         'pos_entropy', 'passive_ratio', 'clause_depth', 'compressibility',
         'char_ngram_repetition', 'zipf_deviation', 'sent_opener_diversity']

EXTRACTORS = {
    'quote_ratio': quote_ratio, 'quote_variety': quote_variety,
    'coord_sub_ratio': coord_sub_ratio, 'function_word_ratio': function_word_ratio,
    'pos_entropy': pos_entropy, 'passive_ratio': passive_ratio,
    'clause_depth': clause_depth, 'compressibility': compressibility,
    'char_ngram_repetition': char_ngram_repetition, 'zipf_deviation': zipf_deviation,
    'sent_opener_diversity': sent_opener_diversity,
}
print('all 11 extractors registered')

all 11 extractors registered


## Compute all eleven, saving incrementally

One article at a time, each extractor wrapped so a single failure yields NaN for that cell only and
never aborts the run. Progress and ETA print every 500 articles; the partial file is rewritten at
the same cadence, so a Kaggle timeout costs time and nothing else (point `RESUME_PATH` at the
partial and re-run).

In [9]:
RESUME_PATH = None      # e.g. '/kaggle/input/aigt-vstat11-partial/vstat11_raw.parquet'
CKPT = f'{OUT_DIR}/vstat11_raw.parquet'

done = {}
if RESUME_PATH:
    prev = pd.read_parquet(find_file(RESUME_PATH, '*.parquet', 'vstat11'))
    done = {r['article_id']: r for _, r in prev.iterrows()}
    print('resuming with', len(done), 'articles already computed')

rows = list(done.values())
todo = df[~df['article_id'].isin(set(done.keys()))].reset_index(drop=True)
print('to compute:', len(todo))

t0 = time.time()
for i, r in todo.iterrows():
    text = r['text']
    rec = {'article_id': r['article_id'], 'label': r['label'],
           'split': r['split'], 'generator': r['generator']}
    for name, fn in EXTRACTORS.items():
        try:
            rec[name] = fn(text)
        except Exception:
            rec[name] = float('nan')
    rows.append(rec)
    if (i + 1) % 500 == 0 or (i + 1) == len(todo):
        pd.DataFrame(rows).to_parquet(CKPT, index=False)
        el = time.time() - t0
        eta = el / (i + 1) * (len(todo) - i - 1) / 60
        print(f'[{i+1:5d}/{len(todo)}] {el/(i+1):.3f}s/doc | ETA {eta:.0f}m', flush=True)

raw11 = pd.DataFrame(rows)
raw11 = raw11.set_index('article_id').loc[df['article_id']].reset_index()   # restore dataset order
raw11.to_parquet(CKPT, index=False)
print('\ndone:', raw11.shape)
print('NaN per feature:')
print(raw11[NEW11].isna().sum().to_string())

to compute: 7101
[  500/7101] 0.006s/doc | ETA 1m
[ 1000/7101] 0.006s/doc | ETA 1m
[ 1500/7101] 0.006s/doc | ETA 1m
[ 2000/7101] 0.006s/doc | ETA 1m
[ 2500/7101] 0.006s/doc | ETA 0m
[ 3000/7101] 0.006s/doc | ETA 0m
[ 3500/7101] 0.006s/doc | ETA 0m
[ 4000/7101] 0.006s/doc | ETA 0m
[ 4500/7101] 0.006s/doc | ETA 0m
[ 5000/7101] 0.006s/doc | ETA 0m
[ 5500/7101] 0.006s/doc | ETA 0m
[ 6000/7101] 0.006s/doc | ETA 0m
[ 6500/7101] 0.006s/doc | ETA 0m
[ 7000/7101] 0.006s/doc | ETA 0m
[ 7101/7101] 0.006s/doc | ETA 0m

done: (7101, 15)
NaN per feature:
quote_ratio              0
quote_variety            0
coord_sub_ratio          0
function_word_ratio      0
pos_entropy              0
passive_ratio            0
clause_depth             0
compressibility          0
char_ngram_repetition    0
zipf_deviation           0
sent_opener_diversity    3


## Sanity — class separation and redundancy against the existing five

Two checks before scaling. First, the raw class means: a feature whose two class means are identical
carries no signal and will show up as inert later, so it is better to see it now. Second, the
correlation of each new feature with each existing one: a new feature correlating above ~0.9 with an
old one is a duplicate, not an addition, and should be flagged in the write-up.

In [10]:
print('raw class means (0 = human, 1 = ai) and the gap:\n')
m = raw11.groupby('label')[NEW11].mean().T
m.columns = ['human', 'ai']
m['gap'] = (m['ai'] - m['human'])
m['rel_gap_%'] = 100 * m['gap'] / m[['human','ai']].abs().max(axis=1).replace(0, np.nan)
print(m.round(4).sort_values('rel_gap_%', key=abs, ascending=False).to_string())

print('\n\ncorrelation with the existing five (|r| > 0.9 means duplicate):\n')
merged_raw = pd.concat([raw11[NEW11].reset_index(drop=True),
                        vs5_al[OLD5].reset_index(drop=True)], axis=1)
corr = merged_raw.corr().loc[NEW11, OLD5]
print(corr.round(2).to_string())
flags = [(a, b, corr.loc[a, b]) for a in NEW11 for b in OLD5 if abs(corr.loc[a, b]) > 0.9]
print('\nredundancy flags:', flags if flags else 'none')

raw class means (0 = human, 1 = ai) and the gap:

                        human      ai     gap  rel_gap_%
quote_ratio            0.1732  0.0780 -0.0952   -54.9598
passive_ratio          0.1224  0.0849 -0.0375   -30.6097
quote_variety          0.9317  0.7431 -0.1886   -20.2409
clause_depth           5.0919  5.6105  0.5185     9.2424
function_word_ratio    0.1764  0.1940  0.0176     9.0554
sent_opener_diversity  0.8608  0.8061 -0.0547    -6.3561
coord_sub_ratio        0.6284  0.6056 -0.0228    -3.6229
zipf_deviation         0.5621  0.5764  0.0143     2.4821
pos_entropy            1.2778  1.3007  0.0228     1.7562
compressibility        0.3331  0.3370  0.0040     1.1735
char_ngram_repetition  0.4055  0.4037 -0.0018    -0.4382


correlation with the existing five (|r| > 0.9 means duplicate):

                       targeted_ppl  burstiness   ttr  entity_density  discourse_coherence
quote_ratio                   -0.00        0.12 -0.08            0.01                -0.07
quote_variety    

## Scale on TRAIN only and write the 16-feature file

The eleven new features get their own StandardScaler and train-median imputation, fit on the train
split alone, exactly as NB4 did for the original five. The existing five are already scaled and are
carried over untouched, so nothing about the current official model changes.

In [11]:
from sklearn.preprocessing import StandardScaler

tr_mask = (raw11['split'] == 'train').to_numpy()
X11 = raw11[NEW11].to_numpy(dtype=np.float64)

train_median = np.nanmedian(X11[tr_mask], axis=0)
Xtr = X11[tr_mask].copy()
ind = np.where(np.isnan(Xtr)); Xtr[ind] = np.take(train_median, ind[1])
scaler11 = StandardScaler().fit(Xtr)

Xall = X11.copy()
ind = np.where(np.isnan(Xall)); Xall[ind] = np.take(train_median, ind[1])
X11s = scaler11.transform(Xall).astype(np.float32)

out = raw11[['article_id', 'label', 'split', 'generator']].copy()
for k, name in enumerate(OLD5):
    out[name] = vs5_al[name].to_numpy(dtype=np.float32)      # already scaled in NB4
for k, name in enumerate(NEW11):
    out[name] = X11s[:, k]

ALL16 = OLD5 + NEW11
assert out[ALL16].isna().sum().sum() == 0, 'NaNs survived imputation'
assert (out['article_id'].to_numpy() == df['article_id'].to_numpy()).all(), 'order broke'

out.to_parquet(f'{OUT_DIR}/vstat16_scaled.parquet', index=False)
with open(f'{OUT_DIR}/scaler11.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler11, 'train_median': train_median,
                 'feature_order': NEW11, 'pos_source': POS_SOURCE}, f)

print('scaled class means (0 = human, 1 = ai):')
print(out.groupby('label')[ALL16].mean().round(3).T.to_string())
print(f'\nwrote vstat16_scaled.parquet {out.shape} | scaler fit on {int(tr_mask.sum())} train rows only')
print('feature order (contract):', ALL16)
print('upload vstat16_scaled.parquet as the Kaggle dataset `aigt-vstat16` for PH2_NB6f')

scaled class means (0 = human, 1 = ai):
label                      0      1
targeted_ppl           0.031 -0.033
burstiness             0.494 -0.479
ttr                   -0.347  0.319
entity_density         0.040 -0.023
discourse_coherence   -0.187  0.185
quote_ratio            0.299 -0.284
quote_variety          0.227 -0.240
coord_sub_ratio        0.145 -0.128
function_word_ratio   -0.356  0.344
pos_entropy           -0.257  0.256
passive_ratio          0.086 -0.093
clause_depth          -0.102  0.089
compressibility       -0.086  0.082
char_ngram_repetition  0.016 -0.010
zipf_deviation        -0.121  0.105
sent_opener_diversity  0.200 -0.214

wrote vstat16_scaled.parquet (7101, 20) | scaler fit on 5363 train rows only
feature order (contract): ['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence', 'quote_ratio', 'quote_variety', 'coord_sub_ratio', 'function_word_ratio', 'pos_entropy', 'passive_ratio', 'clause_depth', 'compressibility', 'char_ngram_repetition',

## Notes

- **Leakage discipline unchanged:** the new scaler and the imputation medians are fit on the train
  split only and persisted in `scaler11.pkl` for reuse at inference. The original five are carried
  over already-scaled from NB4, so the current official model is bit-for-bit unaffected.
- **Feature-order contract:** `OLD5 + NEW11` in that exact order. Downstream notebooks index by
  column name, but the order is what the thesis tables and any future fusion layer will assume.
- **`pos_source`** is stored in the pickle — state in the methodology whether the CAMeL tagger or the
  word-shape fallback produced `pos_entropy`.
- **Read the redundancy table before the ablation.** A new feature correlating above 0.9 with an
  existing one is not new information; if the search later prefers it over its twin, that is a
  coin-flip, not a finding.
- Adding a feature here does **not** promote it into the official Vstat. Promotion happens only if
  PH2_NB6f's two-stage protocol says so.